In [91]:
import os
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dna_features_viewer import GraphicFeature, GraphicRecord
from Bio import SeqIO

cwd = os.getcwd()
if cwd.endswith('Halocins'):
    os.chdir('../..')
    cwd = os.getcwd()

from src.tree.itol_annotation import itol_labels

In [92]:
sns.set_palette('colorblind')
sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.8)
plt.rcParams['font.family'] = 'Helvetica'

palette = sns.color_palette().as_hex()

data_folder = Path('data/')
assert data_folder.is_dir()

amp_db_folder = data_folder / 'amp_db'
assert amp_db_folder.is_dir()

halocins_folder = amp_db_folder / 'Halocins'
assert halocins_folder.is_dir()

halh4_folder = halocins_folder / 'Halocin_H4'
assert halh4_folder.is_dir()

halc8_folder = halocins_folder / 'Halocin_C8'
assert halc8_folder.is_dir()

halc8_out = Path('data/outputs/amp_db/Halocins/Halocin_C8')
halocins_out = Path('data/outputs/amp_db/Halocins')


## Halocins H4, C8 and S8 hits with taxonomy

### Halocin H4

In [93]:
halh4_hits = pd.read_csv(halh4_folder / 'Halocin_H4_dataset_curated.csv', index_col='id')
print(f'Halocin H4 hits: {len(halh4_hits)}')
halh4_hits.head()

Halocin H4 hits: 22


,assembly_accession,protein_id,evalue,bits,tstart,tend,domain,gtdb_phylum,gtdb_class,gtdb_order,gtdb_family,gtdb_genus,gtdb_species,uniprot_id
id,,,,,,,,,,,,,,
WP_014732703.1@GCF_000306765.2,GCF_000306765.2,WP_014732703.1,7.295000e-233,723,1,359,Archaea,Halobacteriota,Halobacteria,Halobacteriales,Haloferacaceae,Haloferax,Haloferax mediterranei,Q48236
WP_241433888.1@GCF_000337535.1,GCF_000337535.1,WP_241433888.1,2.720000e-100,340,48,347,Archaea,Halobacteriota,Halobacteria,Halobacteriales,Natrialbaceae,Natrialba,Natrialba aegyptia,NaN
WP_008458115.1@GCF_000337175.1,GCF_000337175.1,WP_008458115.1,2.408000e-93,320,1,276,Archaea,Halobacteriota,Halobacteria,Halobacteriales,Natrialbaceae,Natrinema,Natrinema gari,L9YSX7
WP_161605605.1@GCF_000337135.1,GCF_000337135.1,WP_161605605.1,1.911000e-42,172,6,156,Archaea,Halobacteriota,Halobacteria,Halobacteriales,Natrialbaceae,Natrialba,Natrialba chahannaoensis,NaN
WP_155396796.1@GCF_000970305.1,GCF_000970305.1,WP_155396796.1,1.735000e-28,130,69,300,Archaea,Halobacteriota,Methanosarcinia,Methanosarcinales,Methanosarcinaceae,Methanosarcina,Methanosarcina barkeri A,A0A0E3SN68


In [94]:
halh4_hits['domain'].value_counts()

Archaea     18
Bacteria     4
Name: domain, dtype: int64

In [95]:
halh4_genus_counts = pd.DataFrame(
    halh4_hits[['domain', 'gtdb_phylum', 'gtdb_class', 'gtdb_order', 'gtdb_family', 'gtdb_genus']].value_counts(),
    columns=['Halocin H4']
).reset_index().sort_values(
    ['domain', 'Halocin H4', 'gtdb_phylum', 'gtdb_class', 'gtdb_order', 'gtdb_family', 'gtdb_genus'],
    ascending=[True, False, True, True, True, True, True],
).set_index(['domain', 'gtdb_phylum', 'gtdb_class', 'gtdb_order', 'gtdb_family', 'gtdb_genus'])

### Halocin C8

In [96]:
halc8_hits = pd.read_csv(halc8_out / 'sequences' / 'HalC8_GTDB_r214_hits.csv', index_col='id')
halc8_hits = halc8_hits[halc8_hits['gtdb_phylum'].notnull()].copy()
halc8_hits['gtdb_order'] = halc8_hits['gtdb_order'].apply(lambda v: v.replace('_', ' '))
print(f'Halocin C8 hits: {len(halc8_hits)}')
halc8_hits.head()

Halocin C8 hits: 109


,assembly_accession,protein_id,domain,gtdb_phylum,gtdb_class,gtdb_order,gtdb_family,gtdb_genus,gtdb_species,ncbi_organism_name,uniprot_id,length
id,,,,,,,,,,,,
WP_148183473.1@GCF_000008665.1,GCF_000008665.1,WP_148183473.1,Archaea,Halobacteriota,Archaeoglobi,Archaeoglobales,Archaeoglobaceae,Archaeoglobus,Archaeoglobus fulgidus,Archaeoglobus fulgidus DSM 4304,NaN,195
MCS7130230.1@GCA_025058955.1,GCA_025058955.1,MCS7130230.1,Archaea,Halobacteriota,Archaeoglobi,Archaeoglobales,Archaeoglobaceae,WYZ-LMO2,WYZ-LMO2 sp025058955,Archaeoglobaceae archaeon,NaN,263
MCS7131040.1@GCA_025058955.1,GCA_025058955.1,MCS7131040.1,Archaea,Halobacteriota,Archaeoglobi,Archaeoglobales,Archaeoglobaceae,WYZ-LMO2,WYZ-LMO2 sp025058955,Archaeoglobaceae archaeon,NaN,89
WP_139025500.1@GCF_000376445.1,GCF_000376445.1,WP_139025500.1,Archaea,Halobacteriota,Halobacteria,Halobacteriales,Haladaptataceae,Haladaptatus,Haladaptatus paucihalophilus,Haladaptatus paucihalophilus DX253,A0A1M7C1A1,287
WP_082837828.1@GCF_001625445.1,GCF_001625445.1,WP_082837828.1,Archaea,Halobacteriota,Halobacteria,Halobacteriales,Haladaptataceae,Haladaptatus,Haladaptatus sp001625445,Haladaptatus sp. R4,NaN,266


In [97]:
halc8_hits['domain'].value_counts()

Archaea     82
Bacteria    27
Name: domain, dtype: int64

In [98]:
halc8_genus_counts = pd.DataFrame(
    halc8_hits[['domain', 'gtdb_phylum', 'gtdb_class', 'gtdb_order', 'gtdb_family', 'gtdb_genus']].value_counts(),
    columns=['Halocin C8']
).reset_index().sort_values(
    ['domain', 'Halocin C8', 'gtdb_phylum', 'gtdb_class', 'gtdb_order', 'gtdb_family', 'gtdb_genus'],
    ascending=[True, False, True, True, True, True, True],
).set_index(['domain', 'gtdb_phylum', 'gtdb_class', 'gtdb_order', 'gtdb_family', 'gtdb_genus'])

### Halocin S8

In [99]:
hals8_hits = pd.read_csv(halocins_folder / 'iterative_gtdb_search' / 'iterative_x6_results.tsv', sep='\t')
hals8_hits = hals8_hits[hals8_hits['query'] == 'Halocin_S8'].reset_index(drop=True)

hals8_hits['domain'] = hals8_hits['taxlineage'].apply(lambda v: v.split(';')[0].replace('d_', ''))
hals8_hits['gtdb_phylum'] = hals8_hits['taxlineage'].apply(lambda v: v.split(';')[1].replace('p_', ''))
hals8_hits['gtdb_class'] = hals8_hits['taxlineage'].apply(lambda v: v.split(';')[2].replace('c_', ''))
hals8_hits['gtdb_order'] = hals8_hits['taxlineage'].apply(lambda v: v.split(';')[3].replace('o_', ''))
hals8_hits['gtdb_family'] = hals8_hits['taxlineage'].apply(lambda v: v.split(';')[4].replace('f_', ''))
hals8_hits['gtdb_genus'] = hals8_hits['taxlineage'].apply(lambda v: v.split(';')[5].replace('g_', ''))
hals8_hits['gtdb_species'] = hals8_hits['taxlineage'].apply(lambda v: v.split(';')[6].replace('s_', ''))
hals8_hits = hals8_hits.drop(columns=['taxlineage'])

In [100]:
hals8_hits['domain'].value_counts()

Archaea    39
Name: domain, dtype: int64

In [101]:
hals8_genus_counts = pd.DataFrame(
    hals8_hits[['domain', 'gtdb_phylum', 'gtdb_class', 'gtdb_order', 'gtdb_family', 'gtdb_genus']].value_counts(),
    columns=['Halocin S8']
).reset_index().sort_values(
    ['domain', 'Halocin S8', 'gtdb_phylum', 'gtdb_class', 'gtdb_order', 'gtdb_family', 'gtdb_genus'],
    ascending=[True, False, True, True, True, True, True],
).set_index(['domain', 'gtdb_phylum', 'gtdb_class', 'gtdb_order', 'gtdb_family', 'gtdb_genus'])

### Merged dataset

In [102]:
halocins_genus = pd.merge(
    halh4_genus_counts,
    halc8_genus_counts,
    how='outer',
    on=['domain', 'gtdb_phylum', 'gtdb_class', 'gtdb_order', 'gtdb_family', 'gtdb_genus'],
)

halocins_genus = pd.merge(
    halocins_genus,
    hals8_genus_counts,
    how='outer',
    on=['domain', 'gtdb_phylum', 'gtdb_class', 'gtdb_order', 'gtdb_family', 'gtdb_genus'],
).fillna(0).astype(int)

halocins_genus['Total'] = halocins_genus['Halocin H4'] + halocins_genus['Halocin C8'] + halocins_genus['Halocin S8']

halocins_genus = halocins_genus.reset_index().sort_values(
    ['domain', 'gtdb_phylum', 'gtdb_class', 'gtdb_order', 'gtdb_family', 'gtdb_genus', 'Total'],
    ascending=[True, True, True, True, True, True, False],
).set_index(['domain', 'gtdb_phylum', 'gtdb_class', 'gtdb_order', 'gtdb_family', 'gtdb_genus'])

In [103]:
bacterial_halocins = halocins_genus.loc['Bacteria'].copy().drop(columns=['Total'])
bacterial_halocins.to_csv(halocins_out / 'halocins_in_bacteria.csv')
bacterial_halocins

Halocin H4  \
gtdb_phylum    gtdb_class    gtdb_order        gtdb_family              gtdb_genus                      
Actinomycetota Actinomycetia Mycobacteriales   Micromonosporaceae       Glycomyces                  0   
                                               Mycobacteriaceae         Corynebacterium             0   
Bacillota      Bacilli       Bacillales        Bacillaceae B            Rossellomorea               0   
                                               DSM-18226                Niallia                     0   
                                               Planococcaceae           Planococcus                 0   
                             Bacillales D      Amphibacillaceae         Paucisalibacillus           0   
                                               Halobacillaceae          Halobacillus                0   
                                                                        Halobacillus A              0   
                             Bacillales H      Marinococcaceae          Alteribacillus              0   
                                               Salisediminibacteriaceae Alteribacter                0   
                             Paenibacillales   Paenibacillaceae         Paenibacillus               3   
                             RES148            Pasteuriaceae            Pasteuria                   0   
                             Staphylococcales  Staphylococcaceae        Staphylococcus              0   
Bacillota A    Clostridia    Lachnospirales    DSM-24629                Natranaerovirga             1   
                             Tissierellales    Peptoniphilaceae         Anaerococcus                0   
Chloroflexota  Chloroflexia  Chloroflexales    Roseiflexaceae           AL-N1                       0   
                             Thermomicrobiales Thermomicrobiaceae       Nitrolancea                 0   
WOR-3          UBA3072       UBA3072           UBA3072                  JAGLZA01                    0   

                                                                                           Halocin C8  \
gtdb_phylum    gtdb_class    gtdb_order        gtdb_family              gtdb_genus                      
Actinomycetota Actinomycetia Mycobacteriales   Micromonosporaceae       Glycomyces                  1   
                                               Mycobacteriaceae         Corynebacterium             2   
Bacillota      Bacilli       Bacillales        Bacillaceae B            Rossellomorea               1   
                                               DSM-18226                Niallia                     1   
                                               Planococcaceae           Planococcus                 3   
                             Bacillales D      Amphibacillaceae         Paucisalibacillus           1   
                                               Halobacillaceae          Halobacillus                3   
                                                                        Halobacillus A              1   
                             Bacillales H      Marinococcaceae          Alteribacillus              1   
                                               Salisediminibacteriaceae Alteribacter                1   
                             Paenibacillales   Paenibacillaceae         Paenibacillus               0   
                             RES148            Pasteuriaceae            Pasteuria                   1   
                             Staphylococcales  Staphylococcaceae        Staphylococcus              5   
Bacillota A    Clostridia    Lachnospirales    DSM-24629                Natranaerovirga             0   
                             Tissierellales    Peptoniphilaceae         Anaerococcus                1   
Chloroflexota  Chloroflexia  Chloroflexales    Roseiflexaceae           AL-N1                       1   
                             Thermomicrobiales Thermomicrobiaceae       Nitrolancea                 

In [104]:
halh4_genus_counts.loc['Archaea'].sort_index()

Halocin H4
gtdb_phylum         gtdb_class       gtdb_order              gtdb_family             gtdb_genus                
Halobacteriota      Halobacteria     Halobacteriales         Haloarculaceae          Halomicrobium            1
                                                             Haloferacaceae          Haloferax                1
                                                             Natrialbaceae           Halovivax                1
                                                                                     Natrialba                2
                                                                                     Natrinema                1
                    Methanomicrobia  Methanomicrobiales      Methanofollaceae        Methanofollis            1
                    Methanosarcinia  Methanosarcinales       Methanosarcinaceae      Methanolobus             1
                                                                                     Methanosarcina           1
                    Syntropharchaeia ANME-1                  ANME-1                  G60ANME1                 1
                                                                                     JAGXOR01                 1
                                                                                     QEXZ01                   2
Methanobacteriota B Thermococci      Thermococcales          Thermococcaceae         Thermococcus             2
                                                                                     Thermococcus C           1
Thermoplasmatota    Thermoplasmata   Methanomassiliicoccales Methanomethylophilaceae Methanoplasma            1
                                                                                     UBA71                    1